# v13 baseline (local calibration)

**Dataset:** merged-dataset-v13 (v12 without its rotated/flipped twins) | **Arm:** `baseline` | **seed** 1 | **100 ep** | **imgsz 640** | **batch 16** | **workers 2**

Same training recipe as the published v12 runs. v13 numbers are **not** comparable to v12 or
Kaggle numbers, because the validation and test images differ. Compare v13 arms only with v13 arms.

Run all cells. Progress bars are live; the RAM watchdog frees the image cache if memory spikes.


In [ ]:
# === Environment check ===
import os, sys, gc, csv, json, time
from pathlib import Path
import torch, psutil
# Plain tqdm, NOT tqdm.auto: inside a notebook kernel tqdm.auto resolves to the ipywidgets
# frontend, and VS Code then blocks on a "Widgets require us to download supporting files from a
# 3rd party website" dialog (@jupyter-widgets/base). The text bar needs no CDN and no network.
from tqdm import tqdm

print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| cuda", torch.version.cuda)
assert torch.cuda.is_available(), "no CUDA - qml venv needs torch cu130"
print("gpu        :", torch.cuda.get_device_name(0),
      "| cc", torch.cuda.get_device_capability(0),
      "| %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
vm = psutil.virtual_memory()
print("system RAM : %.1f GB total, %.1f GB available (%.0f%% used)"
      % (vm.total/1e9, vm.available/1e9, vm.percent))


In [ ]:
# === Config (exp6b-identical unless overridden) ===
ARM        = "baseline"
SEED       = 1
EPOCHS     = 100
IMGSZ      = 640
BATCH      = 16
WORKERS    = 2
PATIENCE   = 30          # exp6b value - do not change, it defines the early-stop budget
CACHE      = False   # False | "ram" | "disk"; RAM cache is watchdogged below
RAM_LIMIT  = 85 # percent of system RAM; above this the image cache is dropped

HERE   = Path(r"F:\0BScDefense\04_Code\journal_analysis\v12_local")
DATA   = Path(r"f:\0BScDefense\05_Datasets\merged-dataset-v13\data.yaml")
os.chdir(HERE)   # keep weight downloads / stray ultralytics dirs out of nb/
# workers MUST appear in RUN and TAG whenever it is not the 8 that every published arm used.
# Without it the R1-3 worker-control re-run of baseline/seed-0 lands on
# runs/local_baseline_s0_640/ and on the perclass+summary files that build_paper_assets.py
# turns into the paper tables, destroying the measured baseline it is supposed to be compared to.
_W     = "" if WORKERS == 8 else f"_W{WORKERS}"
RUN    = f"v13_{ARM}_s{SEED}_{IMGSZ}{_W}"
# imgsz MUST appear in TAG for non-640 runs: without it a 1024 arm overwrites the 640
# results CSV of the same arm/seed and pollutes the noise_floor glob.
_SZ    = "" if IMGSZ == 640 else f"_{IMGSZ}"
TAG    = f"v13_{ARM}_{EPOCHS}ep_s{SEED}{_SZ}{_W}"
assert DATA.exists(), DATA

NAMES = [
    "bean_angular_leaf_spot", "bean_bean_rust", "bean_healthy",
    "cauliflower_bacterial_spot_rot", "cauliflower_black_rot",
    "cauliflower_downy_mildew", "cauliflower_no_disease",
    "chilli_Curling-Leafs", "chilli_Healthy-Leafs", "chilli_Leaf Spot-Leafs",
    "chilli_Leveillula taurica-Leafs", "chilli_Mozaik Leaf",
    "cucumber_Antraknosa", "cucumber_Bacterial Wilt", "cucumber_Downy Mildew",
    "potato_healthy", "potato_blackleg", "potato_late_blight", "potato_early_blight",
    "tomato_bacterial_spot", "tomato_curl_virus", "tomato_two_spotted_spider_mite",
]
GROUP_B_LESION = {
    "potato_healthy", "cucumber_Antraknosa", "cucumber_Bacterial Wilt",
    "potato_late_blight", "potato_early_blight", "bean_bean_rust",
    "potato_blackleg", "bean_angular_leaf_spot",
}
group_of = lambda c: "B_lesion" if c in GROUP_B_LESION else "A_whole_leaf"
REF = {"ALL": 0.8975, "A_whole_leaf": 0.9695, "B_lesion": 0.7716,
       "recall": 0.8697, "mAP50_95": 0.7268}   # 0.7268 = exp6b 100ep. 0.7083 is exp10 @60ep.
print("run:", RUN)


In [ ]:
# === tqdm epoch bar + RAM cache watchdog ===
# Ultralytics prints a per-batch bar of its own; this adds an OVERALL epoch bar carrying the
# live metrics, and frees the dataset image cache if system RAM crosses RAM_LIMIT.

_bar = {"h": None, "t0": None}

def _ram():
    return psutil.virtual_memory().percent

def _drop_image_cache(trainer, why):
    """Free the ultralytics in-RAM decoded-image cache (dataset.ims) and collect."""
    freed = 0
    for loader in (getattr(trainer, "train_loader", None), getattr(trainer, "test_loader", None)):
        ds = getattr(loader, "dataset", None)
        ims = getattr(ds, "ims", None)
        if ims:
            freed += sum(1 for x in ims if x is not None)
            ds.ims = [None] * len(ims)
            for attr in ("im_hw0", "im_hw"):
                if hasattr(ds, attr):
                    setattr(ds, attr, [None] * len(ims))
            ds.cache = False          # stop it refilling
    gc.collect()
    torch.cuda.empty_cache()
    print(f"\n[ram-watchdog] {why}: dropped {freed} cached images -> {_ram():.0f}% used", flush=True)

def on_train_start(trainer):
    _bar["t0"] = time.time()
    # file=sys.stdout keeps the bar out of the stderr stream, which VS Code paints as an error.
    _bar["h"] = tqdm(total=EPOCHS, desc=f"{ARM} s{SEED}", unit="ep", dynamic_ncols=True,
                     file=sys.stdout)
    if _ram() > RAM_LIMIT:
        _drop_image_cache(trainer, f"RAM {_ram():.0f}% > {RAM_LIMIT}% at start")

def on_fit_epoch_end(trainer):
    m = trainer.metrics or {}
    get = lambda k: m.get(k, float("nan"))
    b = _bar["h"]
    if b is not None:
        b.update(1)
        b.set_postfix({
            "mAP50": f"{get('metrics/mAP50(B)'):.4f}",
            "mAP50-95": f"{get('metrics/mAP50-95(B)'):.4f}",
            "recall": f"{get('metrics/recall(B)'):.4f}",
            "RAM%": f"{_ram():.0f}",
            "VRAM": f"{torch.cuda.memory_reserved()/1e9:.1f}G",
        })
    if _ram() > RAM_LIMIT:
        _drop_image_cache(trainer, f"RAM {_ram():.0f}% > {RAM_LIMIT}%")

def on_train_end(trainer):
    if _bar["h"] is not None:
        _bar["h"].close()
    mins = (time.time() - _bar["t0"]) / 60
    print(f"train wall-clock: {mins:.1f} min ({mins/max(EPOCHS,1):.2f} min/ep)")

print(f"watchdog armed at {RAM_LIMIT}% RAM (now {_ram():.0f}%)")


In [ ]:
# === Model: stock YOLO ===
from ultralytics import YOLO
model = YOLO("yolo11n.pt")
print("params %.3fM" % (sum(p.numel() for p in model.model.parameters())/1e6))


In [ ]:
# === Train ===
for ev, fn in (("on_train_start", on_train_start),
               ("on_fit_epoch_end", on_fit_epoch_end),
               ("on_train_end", on_train_end)):
    model.add_callback(ev, fn)

results = model.train(
    data=str(DATA), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, device=0, amp=True,
    patience=PATIENCE, seed=SEED, workers=WORKERS, cache=CACHE,
    project=str(HERE / "runs"), name=RUN, exist_ok=True,
)


In [ ]:
# === Per-class validation on the v13 val split ===
best = HERE / "runs" / RUN / "weights" / "best.pt"
b = YOLO(str(best)).val(data=str(DATA), split="val", imgsz=IMGSZ, verbose=False,
                        project=str(HERE / "runs"), name=f"{RUN}_val", exist_ok=True).box

rows = []
for i, ci in enumerate(tqdm(b.ap_class_index, desc="per-class", unit="cls", file=sys.stdout)):
    ci = int(ci); name = NAMES[ci] if ci < len(NAMES) else str(ci)
    rows.append([TAG, name, group_of(name), round(float(b.p[i]), 4), round(float(b.r[i]), 4),
                 round(float(b.ap50[i]), 4), round(float(b.ap[i]), 4)])
rows.append([TAG, "ALL", "ALL", round(float(b.mp), 4), round(float(b.mr), 4),
             round(float(b.map50), 4), round(float(b.map), 4)])

res_dir = HERE / "results"; res_dir.mkdir(exist_ok=True)
csv_path = res_dir / f"perclass_{TAG}.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["arm", "class", "group", "precision", "recall", "mAP50", "mAP50_95"])
    w.writerows(rows)
print("SAVED", csv_path)


In [ ]:
# === Report ===
# Reads the CSV rather than the in-memory `rows`, so this cell survives a kernel restart.
import pandas as pd
_d = pd.read_csv(HERE / "results" / f"perclass_{TAG}.csv")
df   = _d[_d["class"] != "ALL"].reset_index(drop=True)
allr = _d[_d["class"] == "ALL"].iloc[0].tolist()
g = df.groupby("group")[["mAP50", "mAP50_95", "recall"]].mean()
summary = {"ALL": allr[5], "recall": allr[4], "mAP50_95": allr[6],
           "A_whole_leaf": float(g.loc["A_whole_leaf", "mAP50"]),
           "B_lesion": float(g.loc["B_lesion", "mAP50"])}
for k, v in summary.items():
    print(f"{k:14s} {v:.4f}")
print(f"{'A-B gap':14s} {summary['A_whole_leaf'] - summary['B_lesion']:.4f}")
(HERE / "results" / f"summary_{TAG}.json").write_text(json.dumps(summary, indent=2))
display(df.sort_values("mAP50").reset_index(drop=True))


In [ ]:
# === Training curve ===
import pandas as pd, matplotlib.pyplot as plt
h = pd.read_csv(HERE / "runs" / RUN / "results.csv")
h.columns = [c.strip() for c in h.columns]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(h["epoch"], h["metrics/mAP50(B)"], label="mAP@50")
ax[0].plot(h["epoch"], h["metrics/mAP50-95(B)"], label="mAP@50-95")
ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].set_title(TAG); ax[0].grid(alpha=.3)
for k in ("train/box_loss", "train/cls_loss", "train/dfl_loss"):
    if k in h: ax[1].plot(h["epoch"], h[k], label=k.split("/")[-1])
ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].set_title("losses"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

peak = h["metrics/mAP50(B)"].idxmax()
print(f"peak mAP50 {h['metrics/mAP50(B)'][peak]:.4f} @ epoch {int(h['epoch'][peak])}/{EPOCHS}"
      f" | last-10 mean {h['metrics/mAP50(B)'].tail(10).mean():.4f}")
